In [10]:
# =========================================================
# 1️⃣ IMPORTS
# =========================================================
import requests
from typing import List

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools.retriever import create_retriever_tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
# =========================================================
# 2️⃣ STATIC RESOURCES
# =========================================================
RESOURCES = """
Company Policies:
- Office hours are 9 AM to 6 PM IST.
- Remote work is allowed on Fridays.

Product Information:
- EduMate is a personalized learning platform.
- EduMate connects students with expert mentors.
"""

# =========================================================
# 3️⃣ LLM
# =========================================================
llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}},
    temperature=0
)

# =========================================================
# 4️⃣ RAG PIPELINE
# =========================================================
raw_docs: List[Document] = [
    Document(page_content="EduMate supports personalized learning paths."),
    Document(page_content="EduMate uses AI-based mentor matching."),
    Document(page_content="EduMate tracks student progress."),
    Document(page_content="Liquibase is used for database schema migration."),
    Document(page_content="Spring Boot is used for backend services."),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_tool = create_retriever_tool(
    retriever=retriever,
    name="knowledge_base_search",
    description="Search internal EduMate knowledge base"
)

# =========================================================
# 5️⃣ WEATHER TOOL
# =========================================================
@tool
def get_weather(city: str) -> str:
    """Get real-time weather for a city."""
    api_key = "056b5b3fc2828a9d2932a4e088bf5041"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        return f"Failed to fetch weather for {city}"
    data = response.json()
    return f"Weather in {city}: {data['weather'][0]['description']}, {data['main']['temp']}°C"

# =========================================================
# 6️⃣ MULTIPLICATION TOOL
# =========================================================
@tool
def multiply(numbers: str) -> str:
    """Multiply two numbers. Input format: a,b"""
    a, b = map(int, numbers.split(","))
    return str(a * b)

# =========================================================
# 7️⃣ CREATE AGENT (NEW WAY)
# =========================================================
agent = create_agent(
    llm,
    tools=[rag_tool, get_weather, multiply],
    system_prompt=f"You are a helpful AI agent.\n\nRESOURCES:\n{RESOURCES}"
)

# =========================================================
# 8️⃣ RUN EXAMPLES
# =========================================================
if __name__ == "__main__":

    print("\n--- RAG QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "How does EduMate work?"}]})
    print(result["messages"][-1].content)

    print("\n--- WEATHER QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Delhi?"}]})
    print(result["messages"][-1].content)

    print("\n--- MULTIPLICATION QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "Multiply 12 and 9"}]})
    print(result["messages"][-1].content)

    print("\n--- RESOURCES + RAG QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "Tell me about EduMate and office hours"}]})
    print(result["messages"][-1].content)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- RAG QUERY ---
EduMate works by combining personalized learning with AI-driven mentorship to enhance student outcomes. Here's how it functions:

1. **Personalized Learning Paths**: The platform adapts to each student's unique needs, creating tailored learning journeys based on their strengths, weaknesses, and goals.

2. **AI-Based Mentor Matching**: EduMate uses advanced algorithms to connect students with expert mentors who best align with their learning style, subject requirements, and academic objectives.

3. **Progress Tracking**: The system continuously monitors student performance, providing real-time insights and adjustments to keep learners on track toward their goals.

By integrating these features, EduMate delivers a dynamic and supportive learning environment that empowers students to succeed.

--- WEATHER QUERY ---
The weather data for Delhi could not be retrieved at this time. This may be due to a temporary issue with the weather service. Would you like me to try again